# Notebook 01b: Taxonomy Definition
This notebook implements the multi-label category mapping from the `Assessments_Contributing Factors / Situations` field. We also use `Assessments.1_Primary Problem` as a secondary primary-category label.

In [ ]:
import pandas as pd
import yaml
import numpy as np
from pathlib import Path

with open('configs/main_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

with open('configs/category_taxonomy_v1.yaml', 'r') as f:
    taxonomy_spec = yaml.safe_load(f)

df = pd.read_parquet(config['paths']['raw_data'])
print(f"Loaded {len(df)} records.")

## 1. Multi-Label Mapping Logic

In [ ]:
def map_categories(factors_str, taxonomy):
    if pd.isna(factors_str):
        return []
    
    factors = [f.strip().lower() for f in str(factors_str).split(';')]
    assigned_categories = set()
    
    for cat, details in taxonomy['categories'].items():
        keywords = [k.lower() for k in details['keywords']]
        if any(any(k in f for k in keywords) for f in factors):
            assigned_categories.add(cat)
            
    return list(assigned_categories)

df['categories'] = df[taxonomy_spec['source_field']].apply(lambda x: map_categories(x, taxonomy_spec))
print("Mapping complete.")

## 2. Binarization and Stats

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer(classes=list(taxonomy_spec['categories'].keys()))
category_matrix = mlb.fit_transform(df['categories'])
category_df = pd.DataFrame(category_matrix, columns=mlb.classes_)

print("Category counts:")
print(category_df.sum())

print(f"\nAverage labels per report: {category_df.sum(axis=1).mean():.2f}")
print(f"Reports with zero labels: {(category_df.sum(axis=1) == 0).sum()} ({ (category_df.sum(axis=1) == 0).mean()*100:.1f}%)")

## 3. Secondary Label: Primary Category

In [ ]:
def map_primary_category(primary_str, taxonomy):
    if pd.isna(primary_str):
        return 'Other'
    
    primary = str(primary_str).lower()
    for cat, details in taxonomy['categories'].items():
        keywords = [k.lower() for k in details['keywords']]
        if any(k in primary for k in keywords):
            return cat
    return 'Other'

df['primary_category'] = df['Assessments.1_Primary Problem'].apply(lambda x: map_primary_category(x, taxonomy_spec))
print("\nPrimary category distribution:")
print(df['primary_category'].value_counts())

## 4. Save Category Targets

In [ ]:
output_df = pd.concat([df[['acn_num_ACN', 'primary_category']], category_df], axis=1)
output_df.to_parquet('data/processed/category_targets.parquet')
print("Category targets saved.")